In [ ]:
# 1. 安裝必要套件
!pip install pandas numpy scikit-learn xgboost shap matplotlib openpyxl

lasso&stepwise(逐步)

In [ ]:
# 2. 載入資料
import pandas as pd
import numpy as np

import sys
from pathlib import Path
_root = Path.cwd().resolve()
for _ in range(10):
    if (_root / "src" / "data_layout.py").exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root / "src"))
from data_layout import resolve_processed_data_dir
_csv_dir = resolve_processed_data_dir(_root)
df = pd.read_csv(_csv_dir / "2025_metrics.csv")
if "Season" in df.columns:
    df = df[df["Season"] == 2025].reset_index(drop=True)

# 3. 準備特徵與目標
target = 'Win Rate'
exclude_cols = [c for c in ['Team', 'Season', target] if c in df.columns]
X = df.drop(columns=exclude_cols)
y = df[target]

# 4. 標準化資料
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# 5. Lasso 進行特徵選擇
from sklearn.linear_model import LassoCV

lasso = LassoCV(cv=5, random_state=42, max_iter=10000)
lasso.fit(X_scaled, y)
coef = pd.Series(lasso.coef_, index=X.columns)
selected_features = coef[coef != 0].index.tolist()
print('Lasso 選出的特徵:', selected_features)

# 6. 雙向逐步法（Bidirectional Elimination）
import statsmodels.api as sm

def stepwise_selection(X, y, initial_list=[], threshold_in=0.05, threshold_out=0.10, verbose=True):
    included = list(initial_list)
    while True:
        changed=False
        # forward step
        excluded = list(set(X.columns) - set(included))
        new_pval = pd.Series(index=excluded)
        for new_col in excluded:
            model = sm.OLS(y, sm.add_constant(pd.DataFrame(X[included + [new_col]]))).fit()
            new_pval[new_col] = model.pvalues[new_col]
        best_pval = new_pval.min()
        if best_pval < threshold_in:
            best_feature = new_pval.idxmin()
            included.append(best_feature)
            changed=True
            if verbose:
                print('Add  {:30} with p-value {:.6}'.format(best_feature, best_pval))
        # backward step
        model = sm.OLS(y, sm.add_constant(pd.DataFrame(X[included]))).fit()
        # use all coefs except intercept
        pvalues = model.pvalues.iloc[1:]
        worst_pval = pvalues.max()
        if worst_pval > threshold_out:
            worst_feature = pvalues.idxmax()
            included.remove(worst_feature)
            changed=True
            if verbose:
                print('Drop {:30} with p-value {:.6}'.format(worst_feature, worst_pval))
        if not changed:
            break
    return included

# 處理Lasso未選出特徵的情況
if len(selected_features) > 0:
    stepwise_features = stepwise_selection(X_scaled[selected_features], y, verbose=True)
else:
    print("Lasso未選出任何特徵，改為使用全特徵進行逐步回歸")
    stepwise_features = stepwise_selection(X_scaled, y, verbose=True)

print('Stepwise 選出的特徵:', stepwise_features)

# 7. SHAP 解釋
import shap
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

# 檢查特徵是否為空
if len(stepwise_features) == 0:
    print("警告：特徵選擇未選出任何特徵，使用所有特徵進行分析")
    stepwise_features = X.columns.tolist()

model = LinearRegression()
model.fit(X_scaled[stepwise_features], y)

explainer = shap.Explainer(model, X_scaled[stepwise_features])
shap_values = explainer(X_scaled[stepwise_features])

shap.summary_plot(shap_values, X_scaled[stepwise_features], show=False)
plt.title('SHAP Summary for Stepwise-selected Features')
plt.show()

# 8. 分組訓練與測試
from sklearn.model_selection import train_test_split

teams = df['Team'].tolist()
np.random.seed(42)
test_teams = np.random.choice(teams, size=5, replace=False)
train_teams = [t for t in teams if t not in test_teams]

X_train = X_scaled[df['Team'].isin(train_teams)][stepwise_features]
y_train = y[df['Team'].isin(train_teams)]
X_test = X_scaled[df['Team'].isin(test_teams)][stepwise_features]
y_test = y[df['Team'].isin(test_teams)]

# 9. XGBoost 訓練與預測
import xgboost as xgb
from sklearn.metrics import r2_score

xgb_model = xgb.XGBRegressor(n_estimators=100, random_state=42)
xgb_model.fit(X_train, y_train)

y_train_pred = xgb_model.predict(X_train)
y_test_pred = xgb_model.predict(X_test)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f'訓練組 R2: {train_r2:.3f}')
print(f'測試組 R2: {test_r2:.3f}')

# 顯示測試組預測結果
result = pd.DataFrame({
    'Team': np.array(test_teams),
    '實際勝率': y_test.values,
    '預測勝率': y_test_pred,
    '誤差': y_test_pred - y_test.values
})
print(result)

# 10. XGBoost SHAP
explainer_xgb = shap.Explainer(xgb_model, X_train)
shap_values_xgb = explainer_xgb(X_test)
shap.summary_plot(shap_values_xgb, X_test, show=False)
plt.title('XGBoost SHAP Summary (Test Set)')
plt.show()

lasso&stepwise(forward_selected)

In [ ]:
# 2. 載入資料
import pandas as pd
import numpy as np

import sys
from pathlib import Path
_root = Path.cwd().resolve()
for _ in range(10):
    if (_root / "src" / "data_layout.py").exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root / "src"))
from data_layout import resolve_processed_data_dir
_csv_dir = resolve_processed_data_dir(_root)
df = pd.read_csv(_csv_dir / "2025_metrics.csv")
if "Season" in df.columns:
    df = df[df["Season"] == 2025].reset_index(drop=True)

# 3. 準備特徵與目標
target = 'Win Rate'
exclude_cols = [c for c in ['Team', 'Season', target] if c in df.columns]
X = df.drop(columns=exclude_cols)
y = df[target]

# 4. 標準化資料
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# 5. Lasso 進行特徵選擇
from sklearn.linear_model import LassoCV

lasso = LassoCV(cv=5, random_state=42, max_iter=10000)
lasso.fit(X_scaled, y)
coef = pd.Series(lasso.coef_, index=X.columns)
selected_features = coef[coef != 0].index.tolist()
print('Lasso 選出的特徵:', selected_features)

# 6. 前進選擇法（Forward Selection）
import statsmodels.api as sm

def forward_selection(X, y, initial_list=[], threshold_in=0.05, verbose=True):
    """
    前進選擇法實現
    Parameters:
        X: 特徵DataFrame
        y: 目標變量
        initial_list: 初始特徵列表
        threshold_in: 特徵進入模型的p值閾值
        verbose: 是否顯示詳細過程
    """
    included = list(initial_list)
    remaining = list(set(X.columns) - set(included))
    
    while remaining:
        best_pval = float('inf')
        best_feature = None
        
        # 測試所有剩餘特徵
        for candidate in remaining:
            # 建立包含候選特徵的臨時特徵集
            temp_features = included + [candidate]
            model = sm.OLS(y, sm.add_constant(X[temp_features])).fit()
            pval = model.pvalues[candidate]
            
            # 記錄最佳候選特徵
            if pval < best_pval:
                best_pval = pval
                best_feature = candidate
                
        # 檢查最佳候選特徵是否滿足進入條件
        if best_pval < threshold_in:
            included.append(best_feature)
            remaining.remove(best_feature)
            if verbose:
                print(f'加入特徵: {best_feature:30} p值: {best_pval:.6f}')
        else:
            # 沒有特徵滿足條件時停止
            if verbose:
                print(f'無符合條件的特徵，停止選擇 (最佳p值: {best_pval:.4f})')
            break
    
    return included

# 處理Lasso未選出特徵的情況
if len(selected_features) > 0:
    forward_features = forward_selection(
        X_scaled[selected_features], 
        y,
        verbose=True
    )
else:
    print("Lasso未選出任何特徵，改為使用全特徵進行前進選擇")
    forward_features = forward_selection(X_scaled, y, verbose=True)

print('前進選擇法選出的特徵:', forward_features)

# 7. SHAP 解釋
import shap
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

# 檢查特徵是否為空
if len(stepwise_features) == 0:
    print("警告：特徵選擇未選出任何特徵，使用所有特徵進行分析")
    stepwise_features = X.columns.tolist()

model = LinearRegression()
model.fit(X_scaled[stepwise_features], y)

explainer = shap.Explainer(model, X_scaled[stepwise_features])
shap_values = explainer(X_scaled[stepwise_features])

shap.summary_plot(shap_values, X_scaled[stepwise_features], show=False)
plt.title('SHAP Summary for Stepwise-selected Features')
plt.show()

# 8. 分組訓練與測試
from sklearn.model_selection import train_test_split

teams = df['Team'].tolist()
np.random.seed(42)
test_teams = np.random.choice(teams, size=5, replace=False)
train_teams = [t for t in teams if t not in test_teams]

X_train = X_scaled[df['Team'].isin(train_teams)][stepwise_features]
y_train = y[df['Team'].isin(train_teams)]
X_test = X_scaled[df['Team'].isin(test_teams)][stepwise_features]
y_test = y[df['Team'].isin(test_teams)]

# 9. XGBoost 訓練與預測
import xgboost as xgb
from sklearn.metrics import r2_score

xgb_model = xgb.XGBRegressor(n_estimators=100, random_state=42)
xgb_model.fit(X_train, y_train)

y_train_pred = xgb_model.predict(X_train)
y_test_pred = xgb_model.predict(X_test)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f'訓練組 R2: {train_r2:.3f}')
print(f'測試組 R2: {test_r2:.3f}')

# 顯示測試組預測結果
result = pd.DataFrame({
    'Team': np.array(test_teams),
    '實際勝率': y_test.values,
    '預測勝率': y_test_pred,
    '誤差': y_test_pred - y_test.values
})
print(result)

# 10. XGBoost SHAP
explainer_xgb = shap.Explainer(xgb_model, X_train)
shap_values_xgb = explainer_xgb(X_test)
shap.summary_plot(shap_values_xgb, X_test, show=False)
plt.title('XGBoost SHAP Summary (Test Set)')
plt.show()